# Do it yourself: Bayesian yes–no decisions with binary data

So far, you've been handed sliders on a machine someone else built. This one hands you the parts. You write the lines of code at the heart of the inference — the visualizations are already wired to whatever you return.

The object is a test or classification with known sensitivity and specifity, but operating under an unknown base rate (prevalence). We are trying to infer the PPV and NPV, which depend on that unknown.

We are going to build the apparatus of Bayesian inference and decision step by step.

## 1 · The data model

- We assume that sensitivity and specificity are known and fixed
- The base rate / prevalence is unknown
- We want to infer the PPV, the positive negative predictive value (same as precision)
- As data, we have a certain number of samples that were predicted as positive or negative. Only the predicted condition is known, we don't know the true labels

Also: $$PPV = \frac{\mathrm{sensitivity} \cdot \mathrm{prevalence}}{\mathrm{sensitivity} \cdot \mathrm{prevalence} + (1 - \mathrm{specificity}) \cdot (1 - \mathrm{prevalence})}$$


In [ ]:
from functools import partial
from typing import Callable

import numpy as np
import scypy.stats as stats

Matplotlib is building the font cache; this may take a moment.


### 1.1 · The generative model and the likelihood

We know thinks in terms of a process that could generate the data that we observe, which will lead us to the right formulation for the likelihood. The guiding question is "if I know the prevalence, how likely it would be to observe the predicted positive and negative counts I have?""

We assume the data as i.i.d.: the generative model samples each datum independently from the same probability distribution. This allows us to focus our thinking into a single datum. What's the chance for this datum to be labeled positive?

To be labeled positive, the datum has to be truly positive and be correctly labeled or be actually negative and be mislabeled. Let's call the whole "probability of being predicted positive": $$\theta =  \mathrm{sensitivity} \cdot \mathrm{prevalence} + (1- \mathrm{specificity}) \cdot (1 - \mathrm{prevalence})$$

Because we know the sensitivity and specificity, if we know $\theta$, then we know the prevalence:

In [ ]:
def prevalence_from_theta(theta: float, *, sensitivity: float, specificity: float) -> float:
    return (theta + specificity - 1) / (sensitivity + specificity - 1)

⚠️ The formula above breaks down if $\mathrm{sensitivity} \cdot \mathrm{specificity}=1$ (division by 0). Can you see why that makes sense? 

How are the labels we observe generated given a value for $\theta$? Each sample is a Bernoulli trial with success probability $\theta$. So, the probability of observing a certain count of positive and negative labels is given by the [binomial distribution](https://en.wikipedia.org/wiki/Binomial_distribution), implemented in [scipy.stats.binom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html).

In [ ]:
def likelihood(theta: float, *, num_positive_labels: int, num_negative_labels: int) -> float:
    # ⚠️ Do it yourself! Write the expression for the likelihood
    return ...

## 2 · The prior

The parameter $\theta$ is a proportion, thus contained in the interval $[0, 1]$. You can pick any prior that reflects how much you know about theta, e.g., an uniform prior. 

You should know that a binomial likelihood with a fixed number of trials has a [conjugate prior](https://en.wikipedia.org/wiki/Conjugate_prior), which makes computing the posterior trivial (and is also easy to interpret, its strength being measured as a number of pseudo-observations). The conjugate prior for the binomial is the [beta distribution](https://en.wikipedia.org/wiki/Beta_distribution), implemented in in [scipy.stats.beta](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.beta.html).

(Incidentally: the uniform distribution for in the interval $[0, 1]$ is also a beta distribution!)

In [ ]:
def prior(theta: float) -> float:
    # ⚠️ Do it yourself! Write the expression for the prior
    return ...

## 3 · Computing the unnormalized posterior

TODO:
- Explain that this is trivial and it's just a pointwise multiplication, following Bayes' theorem


In [ ]:
def get_unnormalized_posterior(
        prior: Callable[[float], float],
        likelihood: Callable[[float], float]
) -> Callable[[float], float]:
    return lambda theta: prior(theta) * likelihood(theta)

## 4 · Normalizing the posterior

TODO:
- Explain that this step, apparently so simple, is _the_ challenge of Bayesian inference
- You don't always the posterior to be normalized (e.g., if you want to compare the odds of two hypotheses)


In [ ]:
def normalize_posterior(raw_posterior: Callable[[float], float]) -> Callable[[float], float]:
    # ⚠️ Do it yourself! Integrate the unnormalized posterior to get the denominator of Bayes' theorem,
    #    sometimes called evidence (because it's used to compare models).
    #    Hint: use numerical integration along theta
    evidence: float = ...
    return lambda theta: raw_posterior(theta) / evidence

## 5 · Obtaining the posterior for the PPV

But wait! This is the posterior for $\theta$ and we are actually interested in the PPV. Converting from one to the other is easy...

TODO:
- Explain that the conversion here is easy because the PPV is a function of $theta$ and vice-versa: each value of $theta$ corresponds to one and only one value of PPV
- Give an intuitive explanation on how the conversion works (its a probability density for a continuous value, so it's not enough to map values into values, we have to preserve continuity by stretching and compressing the distributions appropriately. Present this as a tangencial technicality, don't focus on it)


In [ ]:

def prevalence_posterior_from_theta_posterior(theta_posterior: Callable[[float], float]) -> Callable[[float], float]:
    # TODO: complete this function

## 5 · Assembling and visualizing the whole inference

TODO: let's now assemble ste

In [ ]:
# Assembling the inference
# Picking a prior
# Picking the likelihood
# Computing the unnormalized posterior
# Normalizing the posterior
# Translating into the PPV posterior

### Plotting the components

In [ ]:
# TODO: those should expect a function float -> float instead of an array

def plot_curve(p, y, title, color="#C45A26"):
    """A single curve over [0, 1] — used to eyeball the prior and the likelihood."""
    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.plot(p, y, lw=2.4, color=color)
    ax.fill_between(p, y, color=color, alpha=0.10)
    ax.set_xlim(0, 1); ax.set_ylim(bottom=0)
    ax.set_xlabel("p  (success rate)"); ax.set_title(title)
    fig.tight_layout(); plt.show()


def plot_three(p, prior_d, like_d, posterior):
    """The Step-4 triptych on [0, 1]: prior, likelihood, posterior together."""
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(p, prior_d,  lw=2.2, color="#7C7563", label="prior")
    ax.plot(p, like_d,   lw=2.2, color="#355E92", label="likelihood")
    ax.plot(p, posterior, lw=3.0, color="#C45A26", label="posterior")
    ax.fill_between(p, posterior, color="#C45A26", alpha=0.10)
    ax.set_xlim(0, 1); ax.set_ylim(bottom=0)
    ax.set_xlabel("p  (success rate)"); ax.set_ylabel("density")
    ax.set_title("prior  ·  likelihood  ·  posterior")
    ax.legend(frameon=False)
    fig.tight_layout(); plt.show()


def plot_overlay(p, posterior, post_exact):
    """Your numerical posterior against the closed-form Beta — should coincide."""
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(p, posterior,  lw=4.0, color="#C45A26", alpha=0.55,
            label="numerical (your grid)")
    ax.plot(p, post_exact, lw=1.6, ls="--", color="#1A1A1A",
            label="analytic  Beta(α+k, β+n−k)")
    ax.set_xlim(0, 1); ax.set_ylim(bottom=0)
    ax.set_xlabel("p  (success rate)"); ax.set_ylabel("density")
    ax.set_title("numerical posterior  vs  analytic Beta")
    ax.legend(frameon=False)
    fig.tight_layout(); plt.show()

In [ ]:
# TODO: Add plots for prior, likelihood, theta posterior, and PPV posterior

## 6 · An eaier way

TODO:
- Explain that the scheme above has the advantage to work for any prior, but specifically for the beta prior, there's an easier way
- Explain the basics of conjugacy, keep it at the intuitive level
- Explain how to compute the beta posterior directly from the beta prior and the binomial likelihood


In [ ]:
# TODO:
# - Compute the analystic solution
# - Plot the two posteriors (the one above integrated numerically, and computed analytically) as a verification for the numerical implementation